<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Duckiedrone IMU Architecture

Every IMU message a ROS 2 node receives starts on a chip a few millimeters across, soldered to the flight-controller board. On its way to the node it passes through sensor processing, an attitude estimator, a link between two computers, and a ROS 2 bridge, and each of them changes the data. Following that path explains why the fields in one IMU message have different origins and why a topic named `data_raw` can still contain corrected measurements.

This notebook addresses the following questions:

- Which pieces of software touch the IMU data between the sensor chip and a subscriber?

- What does each one add to, or remove from, the signal?

- Where in the chain is the data still raw, and where is it already filtered?

## The IMU on the Duckiedrone

The IMU is embedded in the flight-controller board, where PX4 firmware reads it over the __[Serial Peripheral Interface (SPI)](https://en.wikipedia.org/wiki/Serial_Peripheral_Interface)__ bus. The Raspberry Pi receives data from PX4 through [Micro Air Vehicle Link (MAVLink)](https://mavlink.io/), a communication protocol used by the flight controller and its companion computer.

This arrangement lets PX4 use the IMU for attitude estimation and flight control while also supplying measurements to applications on the Raspberry Pi. The [MAVROS2 bridge implementation](https://github.com/mavlink/mavros) connects the MAVLink connection to ROS 2, where its IMU plugin publishes `sensor_msgs/Imu` messages on `/mavros/imu/data` and `/mavros/imu/data_raw`. `/data` includes an estimated orientation, while `/data_raw` exposes measurements without that orientation estimate. Both topics contain data that has already undergone processing, as the sensor calibration established during [flight controller configuration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/50-sensor-calibrations.html) is applied upstream of ROS 2.

## The bigger picture

[Figure 1](#figure-1) separates the flight-controller processing from the communication path to a subscriber. Processed sensor measurements feed estimation and control as well as telemetry, so the path branches inside PX4.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; max-width:100%; overflow-x:auto; text-align:left;">
                   SPI                                   MAVLink             ROS 2
             board-dependent                              50 Hz
+----------+                 +-------------------------+         +---------+       +-----------------+
| IMU chip |---------------->| PX4                     |-------->| MAVROS2 |------>| Subscriber node |
+----------+                 | - Sensor processing:    |         +---------+       +-----------------+
                             |   gyro up to 800 Hz,    |
                             |   integrated at 250 Hz  |
                             | - EKF2 estimator        |
                             | - Flight control        |
                             +-------------------------+
                                          |
                                          v
                                     +--------+
                                     | Motors |
                                     +--------+</pre>
  <figcaption>
    Figure 1: IMU data flow from the flight-controller board to a ROS 2 subscriber node. PX4 runs on the flight controller, while MAVROS2 runs on the Raspberry Pi. Rates shown are configuration limits or targets; observed rates depend on the running system.
  </figcaption>
</figure>

The `Subscriber node` represents any ROS 2 node that reads the IMU topic, including the reader developed in [Notebook 7](./7-interfacing-with-imu.ipynb).

On a physical Duckiedrone, PX4 runs on the flight-controller microcontroller under the real-time operating system [NuttX](https://github.com/apache/nuttx), while MAVROS2 runs under Linux on the Raspberry Pi. A __Universal Serial Bus (USB)__ connection carries MAVLink between the boards.

A virtual Duckiedrone instead supplies simulated sensor data to PX4 and uses a __User Datagram Protocol (UDP)__ connection for MAVLink, preserving the downstream interfaces used by the subscriber.

These stages operate at different rates. [Table 1](#table-1) distinguishes values in the shipped [Duckiedrone parameter file](https://github.com/duckietown/book-opmanual-dd24/blob/ente/src/_static/dd24-b-fc-parameters/duckiedrone-px4-v4.params) from rates selected elsewhere in the software.

<a id="table-1"></a>

Table 1: IMU data-flow rates and configuration sources.

| Stage | Rate                                                  | Set by |
| --- | ---                                                     | --- |
| IMU chip read by PX4 | Board- and configuration-dependent     | IMU-driver configuration |
| Maximum processed gyro publication rate | $800\,\mathrm{Hz}$ limit | `IMU_GYRO_RATEMAX` parameter |
| Integrated IMU data | $250\,\mathrm{Hz}$ target               | `IMU_INTEG_RATE` parameter |
| EKF2 attitude estimate | Input- and scheduler-dependent       | PX4 estimator and sensor-pipeline configuration |
| `HIGHRES_IMU` and `ATTITUDE_QUATERNION` streams | $50\,\mathrm{Hz}$ target for each | PX4 onboard MAVLink profile; runtime requests can override it |

PX4 parameters can be inspected through [QGroundControl](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/42-flight-controller-configuration.html#installing-qgroundcontrol), while stream defaults are defined in the [MAVLink implementation](https://github.com/duckietown/PX4-Autopilot/blob/d8c6892a0d16792305bc66b0bd965c4e2b60232f/src/modules/mavlink/mavlink_main.cpp). Receiving a topic at approximately $50\,\mathrm{Hz}$ describes the delivery of messages to that subscriber; it does not establish that the IMU chip samples at $50\,\mathrm{Hz}$ or that every field contains a newly acquired value at each publication.

## From the IMU chip to PX4

The IMU chip reports digital counts along its own sensing axes. PX4 converts those counts into physical units, applies the calibration, and rotates the result into the Duckiedrone's body frame. That body frame is __forward-right-down (FRD)__: $x$ forward, $y$ right, and $z$ down. Relative to the forward-left-up body convention used in [Notebook 3](./3-rotation-representations.ipynb#euler-angles-and-the-rotation-matrix), the $y$ and $z$ directions are reversed.

Calibration addresses systematic measurement errors, including the offsets discussed in [Notebook 2](./2-imu-estimation-and-drift.ipynb#why-the-bias-is-fatal-to-dead-reckoning). Its stored values depend on the individual flight controller and [its configuration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/42-flight-controller-configuration.html). Filtering and integration also affect the signal before transmission, so downstream readings retain the consequences of those processing choices.

## The flight controller's onboard attitude estimator

[Notebook 2](./2-imu-estimation-and-drift.ipynb#why-the-imu-data-matters) introduced EKF2, PX4's state estimator. It propagates attitude using the gyroscope and uses other observations to constrain the estimate. Under suitable motion conditions, accelerometer measurements help constrain tilt through their relationship to gravity.

The resulting orientation is represented by a quaternion, as explained in [Notebook 4](./4-quaternions-and-rotation-representations.ipynb). It describes an estimated relationship between frames, while angular velocity and proper acceleration remain measurement-derived signals.

No sensor on the Duckiedrone measures heading, so yaw can start with an offset and drift. The downward time-of-flight rangefinder supplies information about height rather than a heading constraint. This explains why the presence of an attitude estimator does not eliminate the yaw drift discussed earlier.

## From PX4 to MAVROS2 over MAVLink

PX4 transmits several MAVLink message types over the USB connection to the Raspberry Pi companion computer. In the path considered here, `ATTITUDE_QUATERNION` carries the attitude estimate and body angular velocity, while `HIGHRES_IMU` carries processed IMU measurements. The [`HIGHRES_IMU` implementation](https://github.com/duckietown/PX4-Autopilot/blob/d8c6892a0d16792305bc66b0bd965c4e2b60232f/src/modules/mavlink/streams/HIGHRES_IMU.hpp) derives angular velocity and acceleration from integrated measurements and applies matching estimator bias corrections before making the values available downstream.

On the Raspberry Pi, the [Duckietown MAVROS launcher](https://github.com/duckietown/dt-ros2-interface/blob/ente/launchers/mavros.sh) establishes the physical USB-serial or virtual UDP connection, and the [MAVROS IMU plugin](https://github.com/mavlink/mavros/blob/ros2/mavros/src/plugins/imu.cpp) then constructs ROS 2 messages on the default topic `/mavros/imu/data`.

[Table 2](#table-2) follows the fields of `/mavros/imu/data` through this conversion.

<a id="table-2"></a>

Table 2: Sources and interpretation of fields in `/mavros/imu/data`.

| Field                                                         | Source and interpretation |
| ---                                                           | --- |
| `orientation`                                                 | PX4's estimated attitude, converted to ROS frame conventions. |
| `angular_velocity`                                            | Processed body angular velocity accompanying the attitude message, converted to ROS body components in $\mathrm{rad/s}$. |
| `linear_acceleration`                                         | The latest acceleration cached by the IMU plugin from an IMU message, converted to ROS body components in $\mathrm{m/s^2}$. |
| `header.stamp`                                                | The attitude-message time converted through MAVROS time handling; it does not separately timestamp the cached acceleration. |
| `*_covariance`                                                | In this plugin, populated from configured standard deviations when the quantity is available; these values are not a live copy of EKF uncertainty or a measured sample variance. |

MAVROS publishes `/data` when it handles an attitude message, using the most recently received acceleration. As a result, the fields can describe different acquisition times even though they share one message header. This becomes relevant when comparing signals or estimating lag.

The plugin also publishes `/mavros/imu/data_raw` from incoming IMU measurements. In the `HIGHRES_IMU` path, these measurements have already been processed by PX4, despite the topic name. The topic has no orientation estimate, indicated by `orientation_covariance[0] = -1`.

More generally, a covariance array whose first entry is `-1` marks the associated quantity as unavailable, while an all-zero covariance means that uncertainty is unknown. Availability should be checked before interpreting the corresponding values.

Notably, __MAVROS also changes coordinate conventions__. PX4's FRD body components become ROS __forward-left-up (FLU)__ components: $x$ stays forward, while $y$ points left and $z$ points up. This conversion preserves the $x$ component and reverses the signs of the $y$ and $z$ components of both angular velocity and proper acceleration.

Despite its name, `linear_acceleration` carries the accelerometer's proper acceleration, so a level, stationary Duckiedrone reads about $+9.81\,\mathrm{m/s^2}$ on the FLU $z$ axis, as in [Notebook 1](./1-imu-basics.ipynb#the-accelerometer).

The orientation conversion additionally changes the world convention from PX4's __north-east-down (NED)__ to ROS __east-north-up (ENU)__. This is a fixed coordinate conversion. It does not supply the heading information missing from [the attitude estimator](#the-flight-controllers-onboard-attitude-estimator), so the estimated heading can still have an offset or drift.

## From MAVROS2 to the subscriber node

Once the messages are published on ROS 2 topics, subscribers can access their fields. For example, in [Notebook 7](./7-interfacing-with-imu.ipynb) a subscriber reads `/mavros/imu/data`, extracts the available values, and converts the orientation quaternion into angles for easier interpretation. Calibration, scaling, and frame conversion have already occurred upstream, so applying those same operations again would change the meaning of the measurements.

## Why not a Duckietown Postal Service (DTPS) driver, like the other sensors?

The time-of-flight sensor and camera use drivers on the companion computer and connect to application software through [DTPS](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html) and the relevant bridges. The IMU is attached to the flight controller, where its PX4 driver supplies the local estimation and control processes.

Because PX4 already provides a MAVLink interface for these quantities, MAVROS2 can expose them directly to ROS 2. An additional DTPS publisher would consume and republish that existing feed rather than access the IMU chip directly. The fast stabilization loop remains on the flight controller, while applications on the Raspberry Pi receive the telemetry stream.

This division also guides troubleshooting. PX4's sensor driver, its processing and telemetry configuration, the connection to MAVROS, and ROS publication are separate stages. A silent ROS topic calls for checking those boundaries in sequence; silence alone does not identify which stage failed.

## Summary and next steps

An IMU topic combines quantities with different origins: the orientation is an estimate, while angular velocity and proper acceleration are processed measurements that can also incorporate estimated bias corrections. PX4 performs the sensor-side work, and MAVROS converts the transmitted data into ROS conventions and messages.

The same separation applies to timing. Sensor acquisition, processing, MAVLink transmission, and ROS arrival each have their own rate, and cached fields can have different ages within one message.

[Notebook 6](./6-imu-rates-and-configuration.ipynb) examines the rate and configuration settings in more detail before Notebook 7 uses the resulting ROS interface.

## Further reading

[Using the ECL EKF](https://docs.px4.io/main/en/advanced_config/tuning_the_ecl_ekf.html) explains the EKF2 estimator that this notebook treats at the architectural level.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()